In [ ]:
# ── Colab Setup (skip automatically if running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Install dependencies
    import subprocess
    subprocess.run(['pip', 'install', 'neuralforecast', '-q'], check=True)

    # Clone repo if not already present (from deepar/patchtst sessions)
    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    os.chdir('/content/ECE1508_GenAI/notebooks')

    # Create predictions directory
    os.makedirs('/content/ECE1508_GenAI/data/predictions', exist_ok=True)

    # Upload prediction parquets produced by deepar.ipynb and patchtst.ipynb
    preds_present = all(
        os.path.exists(f'/content/ECE1508_GenAI/data/predictions/{f}')
        for f in ['deepar_preds.parquet', 'patchtst_preds.parquet']
    )
    if not preds_present:
        from google.colab import files as colab_files
        print("Upload deepar_preds.parquet and patchtst_preds.parquet from your local data/predictions/ folder:")
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            dest = f'/content/ECE1508_GenAI/data/predictions/{fname}'
            with open(dest, 'wb') as f:
                f.write(data)
            print(f'  Saved → {dest}')
    else:
        print('Prediction parquets already present — skipping upload.')

print('Setup complete.')

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.models.metrics import compute_all

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
deepar   = pd.read_parquet('../data/predictions/deepar_preds.parquet')
patchtst = pd.read_parquet('../data/predictions/patchtst_preds.parquet')

assert len(deepar) == len(patchtst), \
    f"Prediction lengths differ: DeepAR={len(deepar)}, PatchTST={len(patchtst)}"

print(f"Test bars  : {len(deepar):,}")
print(f"Date range : {deepar['datetime'].min()} → {deepar['datetime'].max()}")
print(f"\nDeepAR columns  : {list(deepar.columns)}")
print(f"PatchTST columns: {list(patchtst.columns)}")

In [ ]:
def get_metrics(df: pd.DataFrame) -> dict:
    return compute_all(
        df['y'].values, df['pred'].values,
        df['lo_80'].values, df['hi_80'].values,
        df['lo_90'].values, df['hi_90'].values,
    )

m_deepar   = get_metrics(deepar)
m_patchtst = get_metrics(patchtst)

summary = pd.DataFrame({'DeepAR': m_deepar, 'PatchTST': m_patchtst}).T
summary.columns = ['RMSE', 'MAE', 'Dir Acc', 'Coverage 80%', 'Coverage 90%', 'Sharpe', 'Max DD']

print("=== Test Set Comparison ===")
print(summary.round(4).to_string())
summary.round(4)

In [ ]:
def strategy_cumret(df: pd.DataFrame) -> np.ndarray:
    return np.cumsum(np.sign(df['pred'].values) * df['y'].values)

def buyhold_cumret(df: pd.DataFrame) -> np.ndarray:
    return np.cumsum(df['y'].values)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(strategy_cumret(deepar),   label='DeepAR long/short',   linewidth=1.3, color='steelblue')
ax.plot(strategy_cumret(patchtst), label='PatchTST long/short', linewidth=1.3, color='darkorange')
ax.plot(buyhold_cumret(deepar),    label='Buy & Hold SPY',      linewidth=1.0, linestyle='--',
        alpha=0.6, color='gray')
ax.axhline(0, color='black', linewidth=0.4)
ax.set_title('Cumulative return: long/short strategy vs Buy & Hold (test set 2024–2025)')
ax.set_xlabel('Test bar index')
ax.set_ylabel('Cumulative return_1h')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
deepar['width_90']   = deepar['hi_90']   - deepar['lo_90']
patchtst['width_90'] = patchtst['hi_90'] - patchtst['lo_90']

n = 500
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(deepar['width_90'].values[:n],   label='DeepAR 90% width',   alpha=0.8, linewidth=0.8, color='steelblue')
ax.plot(patchtst['width_90'].values[:n], label='PatchTST 90% width', alpha=0.8, linewidth=0.8, color='darkorange')
ax.set_title('90% prediction interval width — first 500 test bars (lower = sharper)')
ax.set_xlabel('Test bar index')
ax.set_ylabel('hi_90 − lo_90')
ax.legend()
plt.tight_layout()
plt.show()

print(f"DeepAR   mean 90% interval width : {deepar['width_90'].mean():.6f}")
print(f"PatchTST mean 90% interval width : {patchtst['width_90'].mean():.6f}")

In [ ]:
test_raw  = pd.read_parquet('../data/splits/test.parquet').reset_index(drop=True)
vix_vals  = test_raw['vix_log'].values[:len(deepar)]
med_vix   = np.median(vix_vals)

low_mask  = vix_vals <= med_vix
high_mask = vix_vals > med_vix

def regime_metrics(df: pd.DataFrame, mask: np.ndarray) -> dict:
    sub = df[mask].copy()
    return compute_all(
        sub['y'].values, sub['pred'].values,
        sub['lo_80'].values, sub['hi_80'].values,
        sub['lo_90'].values, sub['hi_90'].values,
    )

regime = pd.DataFrame({
    'DeepAR  — Low VIX':   regime_metrics(deepar,   low_mask),
    'DeepAR  — High VIX':  regime_metrics(deepar,   high_mask),
    'PatchTST — Low VIX':  regime_metrics(patchtst, low_mask),
    'PatchTST — High VIX': regime_metrics(patchtst, high_mask),
}).T

regime.columns = ['RMSE', 'MAE', 'Dir Acc', 'Coverage 80%', 'Coverage 90%', 'Sharpe', 'Max DD']

print(f"=== Regime Analysis  (median vix_log split = {med_vix:.3f}) ===")
print(regime.round(4).to_string())
regime.round(4)